# Module 18 — Week 7: Acquisition Hyperparameter Tuning and Candidate Stability

**Objective.** Compare UCB κ = 0.5, 1.0, 1.5, 2.5 and EI ξ = 0%, 1%, 5%, 10% of observed output standard deviation. Evaluate stability, neural-network validation, cross-model agreement and local evidence before selecting one novel query per function.

**Reproducibility.** This notebook retains the original Gaussian Process settings, random generators, candidate counts and 20×20 scikit-learn MLP architecture. The latter differs from the 16-neuron networks described in previous weeks. The supplied `Data` directory contains observations through Week 5; Week 6 query results are appended in Step 1.

### Imports

In [35]:
import os
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Load observations

Locate the Week 7 input CSVs using the same project-folder search convention as Week 6. No special ZIP extraction directory is required.

In [36]:
from pathlib import Path
from IPython.display import display

# Same loading convention as Week 6: search the capstone project folders.
roots = [Path.cwd(), *Path.cwd().parents]
folders = []
for base in roots:
    folders.extend([
        base / "7. Module 18" / "Data",
        base / "Data",
        base / "6. Module 17" / "updated_data_week_6",
        base / "updated_data_week_6",
        base / "7. Capstone Project" / "7. Module 18" / "Data",
        base / "7. Capstone Project" / "6. Module 17" / "updated_data_week_6",
    ])
folder = next((p for p in folders if all((p / f"function_{i}.csv").is_file() for i in range(1, 9))), None)
if folder is None:
    raise FileNotFoundError(
        "Eight function CSVs not found in Module 18 Data or Week 6 updated data folders. "
        "Start Jupyter from your capstone project folder, as in Week 6."
    )
data = {i: pd.read_csv(folder / f"function_{i}.csv") for i in range(1, 9)}
print("Loaded eight datasets:", ", ".join(f"F{i}: {len(data[i])} rows" for i in range(1, 9)))
display(pd.DataFrame([{"Function": i, "Starting observations": len(data[i]), "Dimensions": data[i].shape[1] - 1} for i in range(1, 9)]).set_index("Function"))

Loaded eight datasets: F1: 16 rows, F2: 16 rows, F3: 21 rows, F4: 36 rows, F5: 26 rows, F6: 26 rows, F7: 36 rows, F8: 46 rows


,Starting observations,Dimensions
Function,,
1,16,2
2,16,2
3,21,3
4,36,4
5,26,4
6,26,5
7,36,6
8,46,8


### Week 6 query results

In [37]:
week_6_points = {1: [0.773956, 0.438878], 2: [0.729513, 0.111999], 3: [0.30826, 0.310001, 0.479247], 4: [0.329887, 0.354827, 0.355151, 0.453774], 5: [0.939869, 0.992261, 0.978815, 0.999723], 6: [0.397519, 0.393747, 0.471443, 0.71138, 0.165385], 7: [0.174885, 0.324058, 0.479146, 0.269144, 0.331135, 0.61449], 8: [0.042609, 0.051269, 0.314729, 0.12723, 0.965221, 0.369, 0.150066, 0.389171]}
week_6_outputs = {1: 9.656222973626152e-41, 2: 0.4718443083527728, 3: -0.015498449969482334, 4: -0.20043060928325174, 5: 7121.1519545556275, 6: -0.37973950049527283, 7: 2.9293698997248208, 8: 9.8405301677974}

### Evaluate previous queries

In [38]:
week_6_results = {}
for i in range(1, 9):
    previous_best = data[i]['output'].max()
    new_output = week_6_outputs[i]
    improvement = new_output - previous_best
    is_new_best = new_output > previous_best
    week_6_results[i] = {'previous_best': previous_best, 'new_output': new_output, 'improvement': improvement, 'new_best': is_new_best}

display(pd.DataFrame.from_dict(week_6_results, orient="index").rename_axis("Function"))

,previous_best,new_output,improvement,new_best
Function,,,,
1,7.710875e-16,9.656223e-41,-7.710875e-16,False
2,7.469623e-01,4.718443e-01,-2.751179e-01,False
3,-1.549845e-02,-1.549845e-02,-3.469447e-17,False
4,-2.004306e-01,-2.004306e-01,-2.775558e-17,False
5,7.121152e+03,7.121152e+03,-9.094947e-13,False
6,-3.797395e-01,-3.797395e-01,-5.551115e-17,False
7,2.929370e+00,2.929370e+00,0.000000e+00,False
8,9.911249e+00,9.840530e+00,-7.071921e-02,False


### Update observations

In [39]:
for i in range(1, 9):
    point = np.asarray(week_6_points[i], dtype=float)
    X = data[i].drop(columns="output").to_numpy(dtype=float)
    if not np.any(np.all(np.isclose(X, point, atol=1e-9, rtol=0), axis=1)):
        new_row = {f"x{j+1}": value for j, value in enumerate(point)}
        new_row["output"] = week_6_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([new_row])], ignore_index=True)


### Gaussian Process models

In [40]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import warnings
from sklearn.exceptions import ConvergenceWarning
gp_models = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    gp_models[i] = gp

display(pd.DataFrame({"Function": list(gp_models), "Kernel": [str(gp_models[i].kernel_) for i in gp_models]}).set_index("Function"))

,Kernel
Function,
1,RBF(length_scale=1e-05)
2,RBF(length_scale=0.0431)
3,RBF(length_scale=0.193)
4,RBF(length_scale=0.436)
5,RBF(length_scale=0.363)
6,RBF(length_scale=0.558)
7,RBF(length_scale=0.3)
8,RBF(length_scale=1.02)


### GP prediction diagnostics

In [41]:
rng = np.random.default_rng(42)
diagnostic_candidates = {}
gp_diagnostics = {}
for i in range(1, 9):
    n_dimensions = data[i].shape[1] - 1
    candidates = rng.random((100000, n_dimensions))
    diagnostic_candidates[i] = candidates
    mean_pred, std_pred = gp_models[i].predict(candidates, return_std=True)
    gp_diagnostics[i] = {'mean': mean_pred, 'std': std_pred}

diagnostic_rows = []
for i in range(1, 9):
    mu, sigma = gp_diagnostics[i]['mean'], gp_diagnostics[i]['std']
    diagnostic_rows.append({'Function': i, 'Predicted minimum': mu.min(), 'Predicted maximum': mu.max(), 'Mean range': np.ptp(mu), 'Maximum uncertainty': sigma.max()})
display(pd.DataFrame(diagnostic_rows).set_index('Function').round(6))

,Predicted minimum,Predicted maximum,Mean range,Maximum uncertainty
Function,,,,
1,-0.000595,-0.000000,0.000595,0.001626
2,-0.082060,0.790480,0.872540,0.254967
3,-0.397807,-0.011796,0.386011,0.078155
4,-35.037686,-0.200263,34.837424,7.410821
5,-1860.984440,7121.156914,8982.141354,2074.517757
6,-2.661556,-0.365623,2.295933,0.519562
7,-0.088449,2.929370,3.017819,0.772077
8,5.104954,9.964094,4.859140,0.657580


### UCB parameter tuning

In [42]:
ucb_kappas = [0.5, 1.0, 1.5, 2.5]
ucb_tuning_results = {}
for i in range(1, 9):
    candidates = diagnostic_candidates[i]
    mean_pred = gp_diagnostics[i]['mean']
    std_pred = gp_diagnostics[i]['std']
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    best_output = y_existing[best_index]
    ucb_tuning_results[i] = {}
    for kappa in ucb_kappas:
        ucb_scores = mean_pred + kappa * std_pred
        idx = np.argmax(ucb_scores)
        point = candidates[idx]
        predicted = mean_pred[idx]
        uncertainty = std_pred[idx]
        score = ucb_scores[idx]
        distance_best = np.linalg.norm(point - best_point)
        distances_existing = np.linalg.norm(X_existing - point, axis=1)
        distance_nearest = distances_existing.min()
        ucb_tuning_results[i][kappa] = {'point': point, 'predicted': predicted, 'uncertainty': uncertainty, 'score': score, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

ucb_rows = []
for i, settings in ucb_tuning_results.items():
    for kappa, result in settings.items():
        ucb_rows.append({'Function': i, 'κ': kappa, 'Predicted output': result['predicted'], 'Uncertainty': result['uncertainty'], 'UCB score': result['score'], 'Distance to best': result['distance_best'], 'Nearest distance': result['distance_nearest']})
display(pd.DataFrame(ucb_rows).set_index(['Function', 'κ']).round(5))

Predicted output  Uncertainty   UCB score  Distance to best  \
Function κ                                                                  
1        0.5          -0.00059      0.00163     0.00022           0.13246   
         1.0          -0.00059      0.00163     0.00103           0.13246   
         1.5          -0.00059      0.00163     0.00184           0.13246   
         2.5          -0.00059      0.00163     0.00347           0.13246   
2        0.5           0.78459      0.08588     0.82753           0.02108   
         1.0           0.75076      0.13017     0.88093           0.03075   
         1.5           0.72313      0.15190     0.95098           0.03592   
         2.5           0.64277      0.19418     1.12821           0.04777   
3        0.5          -0.01874      0.03203    -0.00273           0.28913   
         1.0          -0.03076      0.04934     0.01858           0.34704   
         1.5          -0.04748      0.06321     0.04734           0.40202   
         2.5          -0.06615      0.07243     0.11493           0.47405   
4        0.5          -0.23561      0.17698    -0.14712           0.07749   
         1.0          -0.23561      0.17698    -0.05863           0.07749   
         1.5          -0.23561      0.17698     0.02986           0.07749   
         2.5         -14.48257      6.35284     1.39953           0.92495   
5        0.5        7121.15691      0.02126  7121.16754           0.00000   
         1.0        7121.15691      0.02126  7121.17817           0.00000   
         1.5        7121.15691      0.02126  7121.18880           0.00000   
         2.5        5482.08420    900.50802  7733.35424           0.20528   
6        0.5          -0.38779      0.08542    -0.34508           0.16321   
         1.0          -0.45126      0.16398    -0.28728           0.27365   
         1.5          -0.66822      0.33213    -0.17002           0.52932   
         2.5          -0.88633      0.43484     0.20077           0.67693   
7        0.5           2.92937      0.00001     2.92937           0.00000   
         1.0           2.62024      0.37997     3.00021           0.20806   
         1.5           2.62024      0.37997     3.19019           0.20806   
         2.5           2.45486      0.46080     3.60687           0.23777   
8        0.5           9.94504      0.14622    10.01815           0.34519   
         1.0           9.86401      0.23715    10.10116           0.55039   
         1.5           9.68100      0.38967    10.26551           0.76079   
         2.5           9.68100      0.38967    10.65518           0.76079   

              Nearest distance  
Function κ                      
1        0.5           0.11786  
         1.0           0.11786  
         1.5           0.11786  
         2.5           0.11786  
2        0.5           0.02108  
         1.0           0.03075  
         1.5           0.03592  
         2.5           0.04777  
3        0.5           0.12945  
         1.0           0.17968  
         1.5           0.23313  
         2.5           0.29858  
4        0.5           0.06653  
         1.0           0.06653  
         1.5           0.06653  
         2.5           0.54599  
5        0.5           0.00000  
         1.0           0.00000  
         1.5           0.00000  
         2.5           0.20056  
6        0.5           0.16321  
         1.0           0.27365  
         1.5           0.52449  
         2.5           0.66203  
7        0.5           0.00000  
         1.0           0.20806  
         1.5           0.20806  
         2.5           0.23777  
8        0.5           0.34519  
         1.0           0.48499  
         1.5           0.64205  
         2.5           0.64205

### Expected Improvement tuning

In [43]:
from scipy.stats import norm
xi_proportions = [0.0, 0.01, 0.05, 0.1]
ei_tuning_results = {}
for i in range(1, 9):
    candidates = diagnostic_candidates[i]
    mean_pred = gp_diagnostics[i]['mean']
    std_pred = gp_diagnostics[i]['std']
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    best_output = y_existing[best_index]
    y_std = np.std(y_existing)
    ei_tuning_results[i] = {}
    for xi_prop in xi_proportions:
        xi = xi_prop * y_std
        improvement = mean_pred - best_output - xi
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, std_pred, out=np.zeros_like(improvement), where=std_pred > 0)
            ei_scores = improvement * norm.cdf(Z) + std_pred * norm.pdf(Z)
        ei_scores[std_pred <= 0] = 0.0
        idx = np.argmax(ei_scores)
        point = candidates[idx]
        predicted = mean_pred[idx]
        uncertainty = std_pred[idx]
        score = ei_scores[idx]
        distance_best = np.linalg.norm(point - best_point)
        distances_existing = np.linalg.norm(X_existing - point, axis=1)
        distance_nearest = distances_existing.min()
        ei_tuning_results[i][xi_prop] = {'xi': xi, 'point': point, 'predicted': predicted, 'uncertainty': uncertainty, 'score': score, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

ei_rows = []
for i, settings in ei_tuning_results.items():
    for proportion, result in settings.items():
        ei_rows.append({'Function': i, 'ξ (% std)': f'{proportion:.0%}', 'ξ': result['xi'], 'Predicted output': result['predicted'], 'Uncertainty': result['uncertainty'], 'EI score': result['score'], 'Distance to best': result['distance_best']})
display(pd.DataFrame(ei_rows).set_index(['Function', 'ξ (% std)']).round(5))

ξ  Predicted output  Uncertainty  EI score  \
Function ξ (% std)                                                       
1        0%           0.00000          -0.00059      0.00163   0.00039   
         1%           0.00002          -0.00059      0.00163   0.00039   
         5%           0.00008          -0.00059      0.00163   0.00037   
         10%          0.00016          -0.00059      0.00163   0.00034   
2        0%           0.00000           0.78459      0.08588   0.05631   
         1%           0.00255           0.77256      0.10583   0.05474   
         5%           0.01275           0.76831      0.11195   0.04909   
         10%          0.02550           0.76831      0.11195   0.04262   
3        0%           0.00000          -0.03076      0.04934   0.01299   
         1%           0.00078          -0.03076      0.04934   0.01269   
         5%           0.00391          -0.03473      0.05314   0.01161   
         10%          0.00782          -0.03991      0.05777   0.01043   
4        0%           0.00000          -0.23561      0.17698   0.05441   
         1%           0.08735         -14.48257      6.35284   0.02599   
         5%           0.43675         -14.48257      6.35284   0.02214   
         10%          0.87350         -14.48257      6.35284   0.01804   
5        0%           0.00000        5482.08420    900.50802  12.21730   
         1%          21.21823        5482.08420    900.50802  11.50684   
         5%         106.09116        5482.08420    900.50802   9.01419   
         10%        212.18231        5482.08420    900.50802   6.57601   
6        0%           0.00000          -0.52171      0.22513   0.03612   
         1%           0.00590          -0.52171      0.22513   0.03458   
         5%           0.02952          -0.66822      0.33213   0.02999   
         10%          0.05905          -0.66822      0.33213   0.02531   
7        0%           0.00000           2.62024      0.37997   0.04459   
         1%           0.00772           2.62024      0.37997   0.04301   
         5%           0.03860           2.62024      0.37997   0.03711   
         10%          0.07721           2.62024      0.37997   0.03066   
8        0%           0.00000           9.94504      0.14622   0.07678   
         1%           0.01101           9.94504      0.14622   0.07043   
         5%           0.05504           9.76491      0.32789   0.05405   
         10%          0.11008           9.68100      0.38967   0.04108   

                    Distance to best  
Function ξ (% std)                    
1        0%                  0.13246  
         1%                  0.13246  
         5%                  0.13246  
         10%                 0.13246  
2        0%                  0.02108  
         1%                  0.02556  
         5%                  0.02674  
         10%                 0.02674  
3        0%                  0.34704  
         1%                  0.34704  
         5%                  0.35429  
         10%                 0.38924  
4        0%                  0.07749  
         1%                  0.92495  
         5%                  0.92495  
         10%                 0.92495  
5        0%                  0.20528  
         1%                  0.20528  
         5%                  0.20528  
         10%                 0.20528  
6        0%                  0.35968  
         1%                  0.35968  
         5%                  0.52932  
         10%                 0.52932  
7        0%                  0.20806  
         1%                  0.20806  
         5%                  0.20806  
         10%                 0.20806  
8        0%                  0.34519  
         1%                  0.34519  
         5%                  0.72211  
         10%                 0.76079

### Neural-network ensembles

In [44]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
nn_models = {}
nn_x_scalers = {}
nn_y_scalers = {}
nn_training_losses = {}
n_ensemble = 5
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float).reshape(-1, 1)
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y).ravel()
    models = []
    losses = []
    for seed in range(n_ensemble):
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=seed)
        model.fit(X_scaled, y_scaled)
        models.append(model)
        losses.append(model.loss_)
    nn_models[i] = models
    nn_x_scalers[i] = x_scaler
    nn_y_scalers[i] = y_scaler
    nn_training_losses[i] = losses

display(pd.DataFrame({"Function": list(nn_training_losses), "Mean training loss": [np.mean(nn_training_losses[i]) for i in nn_training_losses]}).set_index("Function").round(6))

,Mean training loss
Function,
1,0.000924
2,0.006460
3,0.000440
4,0.000198
5,0.000276
6,0.000301
7,0.000427
8,0.000392


### Neural-network LOOCV

In [45]:
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np
nn_loocv_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    nn_predictions = []
    baseline_predictions = []
    actual_values = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=42)
        model.fit(X_train_scaled, y_train_scaled)
        pred_scaled = model.predict(X_test_scaled)[0]
        pred = y_scaler.inverse_transform([[pred_scaled]])[0, 0]
        nn_predictions.append(pred)
        baseline_predictions.append(np.mean(y_train))
        actual_values.append(y_test)
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

display(pd.DataFrame.from_dict(nn_loocv_results, orient="index").rename_axis("Function").round(6))

,nn_mae,baseline_mae,beats_baseline
Function,,,
1,0.001776,0.001111,False
2,0.218453,0.237181,True
3,0.060294,0.050129,False
4,1.906639,7.081622,True
5,224.182406,1689.428734,True
6,0.235898,0.497547,True
7,0.376668,0.586378,True
8,0.358344,0.941538,True


### Neural-network candidates

In [46]:
rng_nn = np.random.default_rng(42)
nn_candidates = {}
for i in range(1, 9):
    n_dimensions = data[i].shape[1] - 1
    candidates = rng_nn.random((100000, n_dimensions))
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    ensemble_predictions_scaled = np.array([model.predict(candidates_scaled) for model in nn_models[i]])
    ensemble_predictions = np.array([nn_y_scalers[i].inverse_transform(preds.reshape(-1, 1)).ravel() for preds in ensemble_predictions_scaled])
    mean_predictions = ensemble_predictions.mean(axis=0)
    disagreement = ensemble_predictions.std(axis=0)
    idx = np.argmax(mean_predictions)
    selected_point = candidates[idx]
    predicted_output = mean_predictions[idx]
    model_disagreement = disagreement[idx]
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    distance_best = np.linalg.norm(selected_point - best_point)
    distances_existing = np.linalg.norm(X_existing - selected_point, axis=1)
    distance_nearest = distances_existing.min()
    nn_candidates[i] = {'point': selected_point, 'predicted': predicted_output, 'disagreement': model_disagreement, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

display(pd.DataFrame([{'Function': i, 'NN predicted output': r['predicted'], 'Model disagreement': r['disagreement'], 'Distance to best': r['distance_best'], 'Nearest distance': r['distance_nearest']} for i, r in nn_candidates.items()]).set_index('Function').round(5))

,NN predicted output,Model disagreement,Distance to best,Nearest distance
Function,,,,
1,0.00074,0.00083,0.26643,0.13831
2,0.70612,0.08293,0.02095,0.02095
3,-0.01412,0.00313,0.10071,0.10071
4,-0.00196,0.39878,0.07749,0.06653
5,7103.41318,20.90020,0.00000,0.00000
6,-0.34717,0.03975,0.17371,0.17371
7,2.97740,0.20361,0.19795,0.19795
8,10.26440,0.16498,0.35832,0.35832


### GP and NN comparison

In [47]:
gp_nn_distances = {}
for i in range(1, 9):
    nn_point = nn_candidates[i]['point']
    gp_nn_distances[i] = {'UCB': {}, 'EI': {}}
    for kappa in ucb_kappas:
        ucb_point = ucb_tuning_results[i][kappa]['point']
        distance = np.linalg.norm(nn_point - ucb_point)
        gp_nn_distances[i]['UCB'][kappa] = distance
    for xi_prop in xi_proportions:
        ei_point = ei_tuning_results[i][xi_prop]['point']
        distance = np.linalg.norm(nn_point - ei_point)
        gp_nn_distances[i]['EI'][xi_prop] = distance

display(pd.DataFrame([{'Function': i, 'Closest NN–UCB': min(r['UCB'].values()), 'Closest NN–EI': min(r['EI'].values()), 'Closest NN–GP': min([*r['UCB'].values(), *r['EI'].values()])} for i, r in gp_nn_distances.items()]).set_index('Function').round(5))

,Closest NN–UCB,Closest NN–EI,Closest NN–GP
Function,,,
1,0.33845,0.33845,0.33845
2,0.00220,0.00220,0.00220
3,0.22084,0.27656,0.22084
4,0.00000,0.00000,0.00000
5,0.00000,0.20528,0.00000
6,0.18574,0.38373,0.18574
7,0.17845,0.17845,0.17845
8,0.29943,0.29943,0.29943


### Candidate neighbourhoods

In [48]:
import numpy as np
ucb_settings = [0.5, 1.5, 2.5]
ei_settings = [0.0, 0.05, 0.1]
neighbourhood_results = {}
for i in range(1, 9):
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    neighbourhood_results[i] = {}
    for kappa in ucb_settings:
        point = ucb_tuning_results[i][kappa]['point']
        distances = np.linalg.norm(X_existing - point, axis=1)
        nearest_idx = np.argsort(distances)[:3]
        key = f'UCB kappa={kappa}'
        neighbourhood_results[i][key] = []
        for idx in nearest_idx:
            neighbourhood_results[i][key].append({'distance': distances[idx], 'output': y_existing[idx]})
    for xi_prop in ei_settings:
        point = ei_tuning_results[i][xi_prop]['point']
        distances = np.linalg.norm(X_existing - point, axis=1)
        nearest_idx = np.argsort(distances)[:3]
        key = f'EI xi={xi_prop}'
        neighbourhood_results[i][key] = []
        for idx in nearest_idx:
            neighbourhood_results[i][key].append({'distance': distances[idx], 'output': y_existing[idx]})
    point = nn_candidates[i]['point']
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_idx = np.argsort(distances)[:3]
    neighbourhood_results[i]['NN'] = []
    for idx in nearest_idx:
        neighbourhood_results[i]['NN'].append({'distance': distances[idx], 'output': y_existing[idx]})

neighbour_rows = []
for i, methods in neighbourhood_results.items():
    for method, neighbours in methods.items():
        neighbour_rows.append({'Function': i, 'Candidate': method, 'Nearest distance': neighbours[0]['distance'], 'Nearest output': neighbours[0]['output'], 'Best of 3 nearby': max(n['output'] for n in neighbours)})
display(pd.DataFrame(neighbour_rows).set_index(['Function', 'Candidate']).round(5))

Nearest distance  Nearest output  Best of 3 nearby
Function Candidate                                                        
1        UCB kappa=0.5           0.11786         0.00000           0.00000
         UCB kappa=1.5           0.11786         0.00000           0.00000
         UCB kappa=2.5           0.11786         0.00000           0.00000
         EI xi=0.0               0.11786         0.00000           0.00000
         EI xi=0.05              0.11786         0.00000           0.00000
         EI xi=0.1               0.11786         0.00000           0.00000
         NN                      0.13831         0.00000           0.00000
2        UCB kappa=0.5           0.02108         0.74696           0.74696
         UCB kappa=1.5           0.03592         0.74696           0.74696
         UCB kappa=2.5           0.04777         0.74696           0.74696
         EI xi=0.0               0.02108         0.74696           0.74696
         EI xi=0.05              0.02674         0.74696           0.74696
         EI xi=0.1               0.02674         0.74696           0.74696
         NN                      0.02095         0.74696           0.74696
3        UCB kappa=0.5           0.12945        -0.01645          -0.01645
         UCB kappa=1.5           0.23313        -0.03484          -0.01645
         UCB kappa=2.5           0.29858        -0.03484          -0.01645
         EI xi=0.0               0.17968        -0.03484          -0.01645
         EI xi=0.05              0.19413        -0.03484          -0.01645
         EI xi=0.1               0.20181        -0.03484          -0.01645
         NN                      0.10071        -0.01550          -0.01550
4        UCB kappa=0.5           0.06653        -0.30064          -0.20043
         UCB kappa=1.5           0.06653        -0.30064          -0.20043
         UCB kappa=2.5           0.54599       -16.50716          -6.70209
         EI xi=0.0               0.06653        -0.30064          -0.20043
         EI xi=0.05              0.54599       -16.50716          -6.70209
         EI xi=0.1               0.54599       -16.50716          -6.70209
         NN                      0.06653        -0.30064          -0.20043
5        UCB kappa=0.5           0.00000      7121.15195        7121.15195
         UCB kappa=1.5           0.00000      7121.15195        7121.15195
         UCB kappa=2.5           0.20056      6691.93167        7121.15195
         EI xi=0.0               0.20056      6691.93167        7121.15195
         EI xi=0.05              0.20056      6691.93167        7121.15195
         EI xi=0.1               0.20056      6691.93167        7121.15195
         NN                      0.00000      7121.15195        7121.15195
6        UCB kappa=0.5           0.16321        -0.37974          -0.37974
         UCB kappa=1.5           0.52449        -0.45851          -0.37974
         UCB kappa=2.5           0.66203        -0.45851          -0.37974
         EI xi=0.0               0.35968        -0.37974          -0.37974
         EI xi=0.05              0.52449        -0.45851          -0.37974
         EI xi=0.1               0.52449        -0.45851          -0.37974
         NN                      0.17371        -0.37974          -0.37974
7        UCB kappa=0.5           0.00000         2.92937           2.92937
         UCB kappa=1.5           0.20806         2.92937           2.92937
         UCB kappa=2.5           0.23777         2.92937           2.92937
         EI xi=0.0               0.20806         2.92937           2.92937
         EI xi=0.05              0.20806         2.92937           2.92937
         EI xi=0.1               0.20806         2.92937           2.92937
         NN                      0.19795         2.92937           2.92937
8        UCB kappa=0.5           0.34519         9.91125           9.91125
         UCB kappa=1.5           0.64205         9.82199           9.91125
         UCB kappa=2.5           0.64205         

### Function 5 local refinement

In [49]:
i = 5
X_existing = data[i].drop(columns='output').values.astype(float)
y_existing = data[i]['output'].values.astype(float)
best_idx = np.argmax(y_existing)
best_point = X_existing[best_idx]
best_output = y_existing[best_idx]
rng_local = np.random.default_rng(42)
n_local = 100000
local_candidates = best_point + rng_local.normal(loc=0.0, scale=0.03, size=(n_local, len(best_point)))
local_candidates = np.clip(local_candidates, 0, 1)
dist_existing = np.min(np.linalg.norm(local_candidates[:, None, :] - X_existing[None, :, :], axis=2), axis=1)
valid = dist_existing > 0.005
local_candidates = local_candidates[valid]
dist_existing = dist_existing[valid]
gp_mean, gp_std = gp_models[i].predict(local_candidates, return_std=True)
local_scaled = nn_x_scalers[i].transform(local_candidates)
nn_preds_scaled = np.array([model.predict(local_scaled) for model in nn_models[i]])
nn_preds = np.array([nn_y_scalers[i].inverse_transform(p.reshape(-1, 1)).ravel() for p in nn_preds_scaled])
nn_mean = nn_preds.mean(axis=0)
nn_disagreement = nn_preds.std(axis=0)
gp_rank = np.argsort(np.argsort(-gp_mean))
nn_rank = np.argsort(np.argsort(-nn_mean))
combined_rank = gp_rank + nn_rank
idx = np.argmin(combined_rank)
f5_local_candidate = local_candidates[idx]

print("Function 5 local candidate:", np.round(f5_local_candidate, 6))

print('Final boundary selection is assessed separately against the local-search result in the decision step.')

Function 5 local candidate: [1. 1. 1. 1.]
Final boundary selection is assessed separately against the local-search result in the decision step.


### Final validation and submission

In [50]:
final_candidates = {1: np.array([0.858598, 0.697368]), 2: np.array([0.706079, 0.15803]), 3: np.array([0.45032, 0.561567, 0.490526]), 4: np.array([0.384332, 0.391111, 0.336887, 0.416488]), 5: np.array([1.0, 1.0, 1.0, 1.0]), 6: np.array([0.387946, 0.435172, 0.345126, 0.743513, 0.076825]), 7: np.array([0.054205, 0.21683, 0.421532, 0.346756, 0.385001, 0.543902]), 8: np.array([0.030327, 0.231267, 0.050266, 0.208295, 0.81852, 0.414744, 0.307096, 0.583908])}
validation_rows = []
for i, point in final_candidates.items():
    X_existing = data[i].drop(columns='output').to_numpy(dtype=float)
    correct_dimensions = len(point) == X_existing.shape[1]
    in_bounds = bool(np.all((point >= 0) & (point <= 1)))
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_distance = float(distances.min())
    duplicate = bool(np.any(np.all(np.isclose(X_existing, point, atol=1e-10, rtol=0), axis=1)))
    validation_rows.append({'Function': i, 'Dimensions OK': correct_dimensions, 'In bounds': in_bounds, 'Novel': not duplicate, 'Nearest distance': nearest_distance, 'Selected coordinates': '-'.join(f'{x:.6f}' for x in point)})
validation = pd.DataFrame(validation_rows).set_index('Function')
display(validation)
if not bool(validation[['Dimensions OK', 'In bounds', 'Novel']].all().all()):
    raise ValueError('One or more final candidates failed validation.')
print('\nWEEK 7 SUBMISSION POINTS')
for i, point in final_candidates.items():
    print(f'Function {i}: ' + '-'.join(f'{x:.6f}' for x in point))

,Dimensions OK,In bounds,Novel,Nearest distance,Selected coordinates
Function,,,,,
1,True,True,True,0.117860,0.858598-0.697368
2,True,True,True,0.021076,0.706079-0.158030
3,True,True,True,0.129453,0.450320-0.561567-0.490526
4,True,True,True,0.066534,0.384332-0.391111-0.336887-0.416488
5,True,True,True,0.064222,1.000000-1.000000-1.000000-1.000000
6,True,True,True,0.163215,0.387946-0.435172-0.345126-0.743513-0.076825
7,True,True,True,0.208059,0.054205-0.216830-0.421532-0.346756-0.385001-0...
8,True,True,True,0.345195,0.030327-0.231267-0.050266-0.208295-0.818520-0...



WEEK 7 SUBMISSION POINTS
Function 1: 0.858598-0.697368
Function 2: 0.706079-0.158030
Function 3: 0.450320-0.561567-0.490526
Function 4: 0.384332-0.391111-0.336887-0.416488
Function 5: 1.000000-1.000000-1.000000-1.000000
Function 6: 0.387946-0.435172-0.345126-0.743513-0.076825
Function 7: 0.054205-0.216830-0.421532-0.346756-0.385001-0.543902
Function 8: 0.030327-0.231267-0.050266-0.208295-0.818520-0.414744-0.307096-0.583908
